# snATAC-Express Tutorial
### This notebook demonstrates how to use snATAC-Express to predict gene expression from chromatin accessibility data using machine learning.

## Overview
### snATAC-Express runs in two phases:

1. Phase 1: Initial modeling with all peaks and feature importance ranking
2. Phase 2: Refined modeling using only the most important peaks (top 95%)

## 1. Setup and Imports

In [1]:
# snATAC-Express Tutorial: End-to-End Example Using run_multi_test

import os
import sys
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Add the package to the path
sys.path.append(os.path.abspath('snatac_express'))

# Import the specific functions we need
from snatac_express.scripts.data_preprocessing import (
    load_peak_input, 
    load_gex_input, 
    get_pseudobulk
)

print("✅ Environment ready!")

✅ Environment ready!


## 2. Configuration and Data Paths

In [2]:
# Define configuration with correct project directory
import os

# Set the correct project directory
project_dir = '/home/maggiebrown/projects/snATAC-Express'
print(f"🏠 Project directory: {project_dir}")

config = {
    'input_dir': os.path.join(project_dir, 'example_data', 'input_data'),
    'output_dir': os.path.join(project_dir, 'results', 'tutorial_bach2'),
    'config_yaml': os.path.join(project_dir, 'config.yaml'),
    'gene': 'BACH2'
}

# Create output directory
os.makedirs(config['output_dir'], exist_ok=True)
print(f"📁 Output directory: {config['output_dir']}")

# Check input data
if os.path.exists(config['input_dir']):
    input_files = os.listdir(config['input_dir'])
    print(f"📂 Input files available:")
    for file in input_files:
        print(f"  - {file}")
else:
    print(f"❌ Input directory not found: {config['input_dir']}")

🏠 Project directory: /home/maggiebrown/projects/snATAC-Express
📁 Output directory: /home/maggiebrown/projects/snATAC-Express/results/tutorial_bach2
📂 Input files available:
  - sparse_gex_matrix_colnames.txt
  - group_coverages.csv
  - sparse_gex_matrix_rownames.txt
  - sparse_peak_matrix_colnames.txt
  - sparse_peak_matrix_rownames.txt
  - sparse_gex_matrix.txt.mtx
  - genelist_genebody.txt
  - sparse_peak_matrix.txt.mtx


## 3. Inspect Config File

In [3]:
# View the configuration file. This is the file that contains the parameters for the analysis and may be edited by the user.
print("Configuration file contents:")
print("=" * 50)
with open(config['config_yaml'], 'r') as f:
    print(f.read())

Configuration file contents:
# snATAC-Express Configuration

# General settings
project_name: "snATAC_Express_Analysis"
output_dir: "results"
n_jobs: -1  # Number of parallel jobs (-1 = use all cores)
random_seed: 12345

# Input data paths
input_data:
  sparse_gex_matrix: "sparse_gex_matrix.txt.mtx"
  sparse_peak_matrix: "sparse_peak_matrix.txt.mtx"
  group_coverages: "group_coverages.csv"
  gene_list: "genelist_genebody.txt"
  
# Phase 1 settings (Initial modeling and feature ranking)
phase1:
  # Pseudobulk settings
  pseudobulk:
    replicate: "1"  # Which replicate to use (1 or 2)
    min_cells: 10   # Minimum cells per pseudobulk group
    
  # Peak filtering options: Peaks in at least X% of cells to include.
  peak_filters:
    - name: "all_peaks"
      min_sample_presence: 0.0
    - name: "peaks_10pct"
      min_sample_presence: 0.1
    - name: "peaks_50pct" 
      min_sample_presence: 0.5
  
  # WHICH PEAK FILTER TO USE (set this to 0, 1, or 2)
  # 0 = all_peaks (use all peaks r

## 4. Peak at ATAC-seq Peak Data

In [4]:
# Load ATAC-seq peak matrix
print("🔍 Loading ATAC-seq peak data...")

# Load peak data using the package function
peak_data = load_peak_input('sparse_peak_matrix.txt.mtx', input_dir=config['input_dir'])

print(f"�� Peak matrix shape: {peak_data.shape}")
print(f"📊 Peak data info:")
print(f"  - Number of peaks: {peak_data.shape[0]}")
print(f"  - Number of cells: {peak_data.shape[1]}")
print(f"  - Data types: {peak_data.dtypes.unique()}")
print(f"  - Memory usage: {peak_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Show sample of peak data
print("\n📋 Sample peak data (first 3 peaks, first 3 cells):")
print(peak_data.iloc[:3, :min(3, peak_data.shape[1])])

# Check if this is test data
if peak_data.shape[1] <= 2:
    print(f"\n⚠️  This appears to be test data with only {peak_data.shape[1]} cell(s)")
    print("   The tutorial will continue but results may be limited")

🔍 Loading ATAC-seq peak data...
�� Peak matrix shape: (247, 76453)
📊 Peak data info:
  - Number of peaks: 247
  - Number of cells: 76453
  - Data types: [dtype('uint8')]
  - Memory usage: 18.03 MB

📋 Sample peak data (first 3 peaks, first 3 cells):
                        Pool_8#GTTAACGGTGCTTTAC-1  Pool_8#CCTTATGTCGCAACAT-1  \
chr6:89827394-89827894                          0                          0   
chr6:89827897-89828397                          0                          0   
chr6:89828900-89829400                          0                          0   

                        Pool_8#GCATATATCAAACTCA-1  
chr6:89827394-89827894                          0  
chr6:89827897-89828397                          0  
chr6:89828900-89829400                          0  


## 5. Peak at Gene Expression Data

In [5]:
print("🧬 Loading gene expression data...")

# Only pass the matrix file name and input_dir
gex_data = load_gex_input('sparse_gex_matrix.txt.mtx', input_dir=config['input_dir'])

print(f"📈 Gene expression matrix shape: {gex_data.shape}")
print(f"📊 Gene expression data info:")
print(f"  - Number of genes: {gex_data.shape[0]}")
print(f"  - Number of cells: {gex_data.shape[1]}")
print(f"  - Data types: {gex_data.dtypes.unique()}")

# Check if BACH2 is in the data
bach2_expression = gex_data.loc[gex_data.index == 'BACH2']
if not bach2_expression.empty:
    print(f"\n🎯 BACH2 expression found!")
    print(f"  - Expression values: {bach2_expression.values.flatten()}")
    print(f"  - Mean expression: {bach2_expression.values.mean():.4f}")
    print(f"  - Std expression: {bach2_expression.values.std():.4f}")
    print(f"  - Number cells with BACH2 transcripts: {(bach2_expression != 0).sum().sum()}")


    # Show sample of GEX data
    print("\n📋 Sample GEX data (BACH2, first 3 cells):")
    print(bach2_expression.iloc[:3, :min(20, bach2_expression.shape[1])])

else:
    print(f"\n⚠️  BACH2 not found in gene expression data")
    print(f"Available genes: {list(gex_data.index)}")

🧬 Loading gene expression data...
📈 Gene expression matrix shape: (1, 76453)
📊 Gene expression data info:
  - Number of genes: 1
  - Number of cells: 76453
  - Data types: [dtype('uint8')]

🎯 BACH2 expression found!
  - Expression values: [ 0  0  0 ...  0 16  0]
  - Mean expression: 10.6663
  - Std expression: 21.1203
  - Number cells with BACH2 transcripts: 34057

📋 Sample GEX data (BACH2, first 3 cells):
       Pool_8#GTTAACGGTGCTTTAC-1  Pool_8#CCTTATGTCGCAACAT-1  \
BACH2                          0                          0   

       Pool_8#GCATATATCAAACTCA-1  Pool_8#CCGTGCTGTAGTTGGC-1  \
BACH2                          0                          0   

       Pool_8#CATAACGGTTATGTGG-1  Pool_8#CTGACCAAGTAAGTCC-1  \
BACH2                          0                          0   

       Pool_8#GGATGGCCAAACCTAT-1  Pool_8#GGAGCAAGTCCTTCTC-1  \
BACH2                          0                          0   

       Pool_8#CTCTGTTCAATTAAGG-1  Pool_8#TTAGGCCCATCATGGC-1  \
BACH2              

## 6. Run the Pipeline

In [6]:
# Import the main workflow runner
import sys
import os

print("🚀 Starting snATAC-Express Two-Phase Pipeline...")
print("=" * 50)

# Change to the project directory to ensure relative paths work
original_cwd = os.getcwd()
project_dir = '/home/maggiebrown/projects/snATAC-Express'
os.chdir(project_dir)

try:
    # Set up command line arguments for BOTH phases
    sys.argv = [
        'run_multi_test.py',
        '--config', 'config.yaml',
        '--phase', 'both',  # Run both Phase 1 and Phase 2
        '--gene', 'BACH2'   # Optional: specific gene
    ]
    
    # Import and run the main function
    from snatac_express.scripts.run_multi_test import main as run_workflow
    
    run_workflow()
    print("✅ Two-phase pipeline completed successfully!")
    
except Exception as e:
    print(f"❌ Error during pipeline execution: {e}")
    raise
finally:
    # Change back to original directory
    os.chdir(original_cwd)

2025-06-15 21:11:31,948 - INFO - Starting snATAC-Express workflow
2025-06-15 21:11:31,949 - INFO - Configuration: config.yaml
2025-06-15 21:11:31,949 - INFO - Phase(s) to run: both
2025-06-15 21:11:31,950 - INFO - 
2025-06-15 21:11:31,950 - INFO - PHASE 1: Initial modeling with feature selection
2025-06-15 21:11:31,951 - INFO - ============================================================
2025-06-15 21:11:31,955 - INFO - Loading ATAC peaks...


🚀 Starting snATAC-Express Two-Phase Pipeline...


2025-06-15 21:11:32,096 - INFO - Loading gene expression...
2025-06-15 21:11:32,211 - INFO - Processing 1 genes...
2025-06-15 21:11:32,212 - INFO - Processing gene BACH2
2025-06-15 21:11:33,093 - INFO -   Total peaks: 247
2025-06-15 21:11:33,094 - INFO -   Filtered peaks (≥10% samples): 132
2025-06-15 21:11:33,095 - INFO -   Running linear_regression


Average Score (all peaks): -0.013891743743191886


2025-06-15 21:12:01,448 - INFO -     perm_ranker:
2025-06-15 21:12:01,449 - INFO -       All peaks: R² = -0.0139 (132 peaks)
2025-06-15 21:12:01,449 - INFO -       95% peaks: R² = 0.4249 (79 peaks)


...

Average Score (95% peaks): 0.5536427855990643
Average Score (all peaks): 0.6505049705505371


2025-06-15 21:25:37,500 - INFO -     xgb_ranker: R² = 0.6505 (116 peaks)
2025-06-15 21:25:37,501 - INFO -   Running lightgbm


Average Score (95% peaks): 0.613093888759613
Average Score (all peaks): 0.5872345945284504


2025-06-15 21:25:46,439 - INFO -     lgbm_ranker: R² = 0.5872 (116 peaks)
2025-06-15 21:25:46,440 - INFO - Step 5: Creating Phase 2 master aggregated peak ranks
2025-06-15 21:25:46,440 - INFO - Creating Phase 2 master aggregated peak ranks file...
2025-06-15 21:25:46,442 - INFO -   BACH2: 116 peaks for Phase 2
2025-06-15 21:25:46,445 - INFO -   Saved Phase 2 master aggregated peak ranks to results/aggregated_results/master_aggregated_peak_ranks.csv
2025-06-15 21:25:46,447 - INFO -   Saved Phase 2 aggregation summary to results/aggregated_results/phase2_aggregation_summary.txt
2025-06-15 21:25:46,447 - INFO - Step 6: Summarizing Phase 2 results
2025-06-15 21:25:46,450 - INFO - Saved Phase 2 summary to results/phase2_cv_summary.txt
2025-06-15 21:25:46,450 - INFO - 
Phase 2 Summary Statistics:
2025-06-15 21:25:46,451 - INFO - Total genes analyzed: 1
2025-06-15 21:25:46,452 - INFO - Average R²: 0.4693
2025-06-15 21:25:46,452 - INFO - Median R²: 0.5592
2025-06-15 21:25:46,454 - INFO - 
Phas

Average Score (95% peaks): 0.5984691239161598
✅ Two-phase pipeline completed successfully!


In [8]:
import os
import glob

# Check what's in the results directory
print("=== CHECKING RESULTS STRUCTURE ===")

# Check if results directory exists
if os.path.exists("results"):
    print("\nContents of results/:")
    for item in os.listdir("results"):
        print(f"  {item}")
    
    # Check phase1_results
    if os.path.exists("results/phase1_results"):
        print("\nContents of results/phase1_results/:")
        for item in os.listdir("results/phase1_results"):
            print(f"  {item}")
            
        # Check BACH2 directory
        if os.path.exists("results/phase1_results/BACH2"):
            print("\nContents of results/phase1_results/BACH2/:")
            for root, dirs, files in os.walk("results/phase1_results/BACH2"):
                level = root.replace("results/phase1_results/BACH2", '').count(os.sep)
                indent = ' ' * 2 * level
                print(f"{indent}{os.path.basename(root)}/")
                subindent = ' ' * 2 * (level + 1)
                for file in files[:5]:  # Show first 5 files
                    print(f"{subindent}{file}")
else:
    print("No results directory found!")
    
# Check current working directory
print(f"\nCurrent working directory: {os.getcwd()}")

# Look for any BACH2 results anywhere
print("\n=== SEARCHING FOR BACH2 FILES ===")
bach2_files = glob.glob("**/BACH2/**/*.csv", recursive=True)
print(f"Found {len(bach2_files)} CSV files related to BACH2:")
for f in bach2_files[:10]:  # Show first 10
    print(f"  {f}")

=== CHECKING RESULTS STRUCTURE ===

Contents of results/:
  aggregated
  tutorial_bach2
  snATAC_Express_20250612_210229.log
  snATAC_Express_20250612_210521.log
  snATAC_Express_20250612_210121.log
  phase1
  phase2

Current working directory: /home/maggiebrown

=== SEARCHING FOR BACH2 FILES ===


KeyboardInterrupt: 

## 10. Feature Importance Analysis

In [ ]:
# Analyze feature importance for the best model
print("🔍 Analyzing feature importance...")

# Get the best model based on R² score
best_model_name = max(results.keys(), key=lambda x: results[x]['r2_score'])
print(f"🏆 Best model: {best_model_name} (R² = {results[best_model_name]['r2_score']:.4f})")

# Get feature importance from the best model
if best_model_name in ['RF', 'XGB', 'LGBM']:
    # Get feature importance
    best_model = model_builder.models[best_model_name]
    if hasattr(best_model, 'feature_importances_'):
        importance = best_model.feature_importances_
    else:
        # For cross-validated models, we might need to retrain
        print("⚠️  Feature importance not available for cross-validated model")
        importance = None
    
    if importance is not None:
        # Create feature importance dataframe
        importance_df = pd.DataFrame({
            'Feature': selected_features,
            'Importance': importance
        }).sort_values('Importance', ascending=False)
        
        # Plot top features
        plt.figure(figsize=(12, 8))
        top_features = importance_df.head(20)
        plt.barh(range(len(top_features)), top_features['Importance'])
        plt.yticks(range(len(top_features)), top_features['Feature'])
        plt.xlabel('Feature Importance')
        plt.title(f'Top 20 Most Important Features - {best_model_name}')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.savefig(os.path.join(config['output_dir'], 'feature_importance.png'), dpi=300, bbox_inches='tight')
        plt.show()
        
        print(f"�� Feature importance plot saved to: {config['output_dir']}/feature_importance.png")
        print(f"\n🏆 Top 10 most important features:")
        for i, (_, row) in enumerate(importance_df.head(10).iterrows()):
            print(f"  {i+1}. {row['Feature']} (importance: {row['Importance']:.4f})")
    else:
        print("⚠️  Feature importance analysis not available")
else:
    print(f"⚠️  Feature importance analysis not available for {best_model_name}")

## 11. Save Results and Summary

In [ ]:
# Save results and create summary
print("💾 Saving results and creating summary...")

# Save model results
results_df = pd.DataFrame(results).T
results_df.to_csv(os.path.join(config['output_dir'], 'model_results.csv'))
print(f"📊 Model results saved to: {config['output_dir']}/model_results.csv")

# Save selected features
features_df = pd.DataFrame({'Feature': selected_features})
features_df.to_csv(os.path.join(config['output_dir'], 'selected_features.csv'), index=False)
print(f"🎯 Selected features saved to: {config['output_dir']}/selected_features.csv")

# Create summary report
summary = f"""
# snATAC-Express Tutorial Results Summary

## Configuration
- Target Gene: {config['gene']}
- Input Directory: {config['input_dir']}
- Output Directory: {config['output_dir']}
- Models Tested: {', '.join(config['models'])}
- Cross-validation Folds: {config['cv_folds']}

## Data Summary
- Peak Matrix Shape: {peak_data.shape}
- Gene Expression Matrix Shape: {gex_data.shape}
- Pseudobulk Data Shape: {pseudobulk_data.shape}
- Selected Features: {len(selected_features)}

## Model Performance
"""
for model_name, metrics in results.items():
    summary += f"""
### {model_name}
- R² Score: {metrics['r2_score']:.4f}
- MAE: {metrics['mae']:.4f}
- RMSE: {metrics['rmse']:.4f}
"""

summary += f"""
## Best Model
- Model: {best_model_name}
- R² Score: {results[best_model_name]['r2_score']:.4f}

## Files Generated
- model_results.csv: Detailed model performance metrics
- selected_features.csv: List of selected features
- model_performance.png: Performance comparison plots
- feature_importance.png: Feature importance analysis (if available)

Generated on: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

# Save summary
with open(os.path.join(config['output_dir'], 'summary_report.md'), 'w') as f:
    f.write(summary)

print(f"📝 Summary report saved to: {config['output_dir']}/summary_report.md")

# Display summary
print("\n" + "="*60)
print("�� TUTORIAL COMPLETED SUCCESSFULLY!")
print("="*60)
print(summary)
print("="*60)

## 12. Advance Analysis and Visualation

In [ ]:
# Optional: Additional analysis and visualization
print("🔬 Performing additional analysis...")

# Correlation analysis between peaks and target gene
if 'BACH2' in gex_data.index:
    target_expression = gex_data.loc['BACH2']
    
    # Calculate correlations
    correlations = []
    for feature in selected_features[:20]:  # Top 20 features
        if feature in pseudobulk_data.columns:
            corr = pseudobulk_data[feature].corr(target_expression)
            correlations.append({'Feature': feature, 'Correlation': corr})
    
    corr_df = pd.DataFrame(correlations).sort_values('Correlation', key=abs, ascending=False)
    
    # Plot correlations
    plt.figure(figsize=(12, 8))
    plt.barh(range(len(corr_df)), corr_df['Correlation'])
    plt.yticks(range(len(corr_df)), corr_df['Feature'])
    plt.xlabel('Correlation with BACH2 Expression')
    plt.title('Top 20 Features: Correlation with BACH2 Expression')
    plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(config['output_dir'], 'correlation_analysis.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"💾 Correlation analysis saved to: {config['output_dir']}/correlation_analysis.png")
    print(f"\n🔗 Top 5 most correlated features:")
    for i, (_, row) in enumerate(corr_df.head(5).iterrows()):
        print(f"  {i+1}. {row['Feature']} (correlation: {row['Correlation']:.4f})")

else:
    print("⚠️  Correlation analysis skipped (BACH2 not found in expression data)")

print("\n✅ Additional analysis completed!")